<a href="https://colab.research.google.com/github/gurudattamanpreet/Practice/blob/main/Titanic5_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from sklearn.utils import class_weight

In [3]:
df=pd.read_csv('https://raw.githubusercontent.com/gurudattamanpreet/datasets/refs/heads/main/Titanic.csv')

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [6]:
df.drop(['PassengerId','Ticket','Cabin','Name'],axis=1,inplace=True)

In [7]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [8]:
df.fillna({'Age':df['Age'].mean(),'Embarked':df['Embarked'].mode()[0]},inplace=True)

In [9]:
df.isna().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,0
SibSp,0
Parch,0
Fare,0
Embarked,0


In [10]:
pd.DataFrame({'dtypes':df.dtypes,'unique_values':df.nunique()})

,dtypes,unique_values
Survived,int64,2
Pclass,int64,3
Sex,object,2
Age,float64,89
SibSp,int64,7
Parch,int64,7
Fare,float64,248
Embarked,object,3


In [11]:
df['Familysize']=df['SibSp']+df['Parch']+1
df['Alone']=(df['Familysize']==1).astype(int)
df['Fareperperson']=df['Fare']/df['Familysize']

In [12]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Familysize,Alone,Fareperperson
0,0,3,male,22.0,1,0,7.2500,S,2,0,3.62500
1,1,1,female,38.0,1,0,71.2833,C,2,0,35.64165
2,1,3,female,26.0,0,0,7.9250,S,1,1,7.92500
3,1,1,female,35.0,1,0,53.1000,S,2,0,26.55000
4,0,3,male,35.0,0,0,8.0500,S,1,1,8.05000


In [13]:
df.drop(['SibSp','Parch'],axis=1,inplace=True)

In [14]:
df.head()

,Survived,Pclass,Sex,Age,Fare,Embarked,Familysize,Alone,Fareperperson
0,0,3,male,22.0,7.2500,S,2,0,3.62500
1,1,1,female,38.0,71.2833,C,2,0,35.64165
2,1,3,female,26.0,7.9250,S,1,1,7.92500
3,1,1,female,35.0,53.1000,S,2,0,26.55000
4,0,3,male,35.0,8.0500,S,1,1,8.05000


In [15]:
df['Sex']=df['Sex'].map({'male':1,'female':0})

In [17]:
df=pd.get_dummies(df,columns=['Embarked'],drop_first=True)

In [18]:
df.head()

,Survived,Pclass,Sex,Age,Fare,Familysize,Alone,Fareperperson,Embarked_Q,Embarked_S
0,0,3,1,22.0,7.2500,2,0,3.62500,False,True
1,1,1,0,38.0,71.2833,2,0,35.64165,False,False
2,1,3,0,26.0,7.9250,1,1,7.92500,False,True
3,1,1,0,35.0,53.1000,2,0,26.55000,False,True
4,0,3,1,35.0,8.0500,1,1,8.05000,False,True


In [19]:
X=df.drop(['Survived'],axis=1)
y=df['Survived']

In [21]:
sc=StandardScaler()
X_Scaled=sc.fit_transform(X)

In [23]:
X_Scaled.shape

(891, 9)

In [24]:
X_train,X_test,y_train,y_test=train_test_split(X_Scaled,y,test_size=0.2,random_state=1,stratify=y)

In [27]:
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # binary output

# Step 3: Compile and Train
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100, batch_size=16, validation_split=0.2)

# Step 4: Evaluate
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6080 - loss: 0.6560 - val_accuracy: 0.7692 - val_loss: 0.5562
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7936 - loss: 0.5242 - val_accuracy: 0.7762 - val_loss: 0.4837
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8007 - loss: 0.4733 - val_accuracy: 0.7622 - val_loss: 0.4649
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8213 - loss: 0.4213 - val_accuracy: 0.7692 - val_loss: 0.4715
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7957 - loss: 0.4551 - val_accuracy: 0.7692 - val_loss: 0.4715
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8159 - loss: 0.4250 - val_accuracy: 0.7762 - val_loss: 0.4762
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8242 - loss: 0.4185 - val_accuracy: 0.7622 - val_loss: 0.4712
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8019 - loss: 0.4469 - val_accuracy: 0.7762 - 

In [28]:
from sklearn.metrics import recall_score

# Step 1: Get predicted labels (0 or 1)
y_pred_test = (model.predict(X_test) > 0.5).astype("int32")  # threshold 0.5 by default

# Step 2: Compute recall
recall = recall_score(y_test, y_pred_test)
print("Recall (Test):", recall)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
Recall (Test): 0.7246376811594203


In [37]:
from sklearn.utils import class_weight
import numpy as np

# Calculate class weights
class_weights = class_weight.compute_class_weight(class_weight='balanced',
                                                  classes=np.unique(y_train),
                                                  y=y_train)

# Convert to dict
class_weight_dict = dict(zip(np.unique(y_train), class_weights))

In [38]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(1, activation='sigmoid'),
])

model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Recall()])

callbacks = [
    EarlyStopping(monitor='val_recall', patience=10, mode='max', restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_recall', factor=0.5, patience=5, mode='max')
]

model.fit(X_train, y_train,
          epochs=200, batch_size=16,
          validation_split=0.2,
          class_weight=class_weight_dict,
          callbacks=callbacks)

Epoch 1/200


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.4836 - loss: 0.9254 - recall_3: 0.4985 - val_accuracy: 0.7413 - val_loss: 0.6480 - val_recall_3: 0.7447 - learning_rate: 0.0010
Epoch 2/200
31/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6558 - loss: 0.7261 - recall_3: 0.6606

/usr/local/lib/python3.11/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_recall` which is not available. Available metrics are: accuracy,loss,recall_3,val_accuracy,val_loss,val_recall_3
  current = self.get_monitor_value(logs)
/usr/local/lib/python3.11/dist-packages/keras/src/callbacks/callback_list.py:145: UserWarning: Learning rate reduction is conditioned on metric `val_recall` which is not available. Available metrics are: accuracy,loss,recall_3,val_accuracy,val_loss,val_recall_3,learning_rate.
  callback.on_epoch_end(epoch, logs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6599 - loss: 0.7197 - recall_3: 0.6667 - val_accuracy: 0.7692 - val_loss: 0.6203 - val_recall_3: 0.7872 - learning_rate: 0.0010
Epoch 3/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6881 - loss: 0.6177 - recall_3: 0.6975 - val_accuracy: 0.7692 - val_loss: 0.5923 - val_recall_3: 0.7660 - learning_rate: 0.0010
Epoch 4/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7148 - loss: 0.6144 - recall_3: 0.7129 - val_accuracy: 0.7692 - val_loss: 0.5710 - val_recall_3: 0.7660 - learning_rate: 0.0010
Epoch 5/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7357 - loss: 0.5344 - recall_3: 0.7494 - val_accuracy: 0.7622 - val_loss: 0.5487 - val_recall_3: 0.7660 - learning_rate: 0.0010
Epoch 6/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7772 - loss: 0.5003 - recall_3: 0.7973 - val_accuracy: 0.7552 - val_loss: 0.5415 - val_recall_3: 0.7660 - learning_rate: 0.0010
Epoch 7/200
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/s

In [39]:
from sklearn.metrics import classification_report

y_probs = model.predict(X_test)
y_pred = (y_probs > 0.5).astype("int32")

print(classification_report(y_test, y_pred))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
              precision    recall  f1-score   support

           0       0.88      0.83      0.85       110
           1       0.75      0.81      0.78        69

    accuracy                           0.82       179
   macro avg       0.81      0.82      0.81       179
weighted avg       0.83      0.82      0.82       179



In [40]:
y_probs = model.predict(X_train)
y_pred = (y_probs > 0.5).astype("int32")

print(classification_report(y_train, y_pred))

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
              precision    recall  f1-score   support

           0       0.90      0.83      0.86       439
           1       0.76      0.85      0.80       273

    accuracy                           0.84       712
   macro avg       0.83      0.84      0.83       712
weighted avg       0.85      0.84      0.84       712



In [ ]:
accuracy_sc